In [ ]:
# =========================
# nb_ads_load
# =========================
# Phase 8, SC-12: Load tpir_load_equivalent to external Asset Data Store (ADS) via REST API
# Handles batching, retries, and audit logging
# WARNING: Uses requests library - ensure it's available in Fabric cluster

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
import json
import sys
import time

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    import requests
except ImportError:
    print("[ERROR] requests library not available in this Fabric environment")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_ads_load] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

# Verify tables exist
try:
    if not spark.catalog.tableExists("tpir_load_equivalent"):
        print("[ERROR] tpir_load_equivalent table does not exist")
        sys.exit(1)
    if not spark.catalog.tableExists("ads_load_audit"):
        print("[ERROR] ads_load_audit table does not exist")
        sys.exit(1)
except Exception as e:
    print(f"[ERROR] Failed to verify tables: {e}")
    sys.exit(1)

# ---------- Load ADS config ----------
def load_ads_config():
    """Load ADS configuration from ads_config.json."""
    try:
        path = "/lakehouse/default/Files/config/ads_config.json"
        raw = mssparkutils.fs.head(path, 500_000)
        config = json.loads(raw)
        print(f"[nb_ads_load] Loaded ADS config: endpoint={config.get('base_url')}")
        return config
    except FileNotFoundError:
        print("[ERROR] ads_config.json not found - cannot proceed")
        return None
    except json.JSONDecodeError as e:
        print(f"[ERROR] ads_config.json is malformed JSON: {e}")
        return None
    except Exception as e:
        print(f"[ERROR] Failed to load ads_config.json: {type(e).__name__}: {e}")
        return None

ads_cfg = load_ads_config()
if ads_cfg is None:
    print("[nb_ads_load] Config load failed; exiting")
    mssparkutils.notebook.exit("CONFIG_ERROR")

ads_endpoint = ads_cfg.get("base_url", "").strip()
if not ads_endpoint:
    print("[ERROR] ADS base_url not configured")
    mssparkutils.notebook.exit("CONFIG_ERROR")

api_version = ads_cfg.get("api_version", "v1").strip()
batch_size = int(ads_cfg.get("batch_size", 500))
timeout_sec = int(ads_cfg.get("timeout_seconds", 60))
retry_max = int(ads_cfg.get("retry_max_attempts", 3))
retry_backoff = int(ads_cfg.get("retry_backoff_seconds", 5))

if batch_size <= 0 or timeout_sec <= 0 or retry_max <= 0:
    print("[ERROR] Invalid config values: batch_size, timeout_sec, retry_max must be > 0")
    mssparkutils.notebook.exit("CONFIG_ERROR")

print(f"[nb_ads_load] Config: endpoint={ads_endpoint}, batch_size={batch_size}, retry_max={retry_max}")

# ---------- Load TPIR data ----------
try:
    tpir = spark.table("tpir_load_equivalent").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    if tpir.rdd.isEmpty():
        print(f"[nb_ads_load] No tpir_load_equivalent rows for period={period}, run_id={run_id}")
        spark.sql(f"""
            INSERT INTO ads_load_audit
            (period, run_id, dfm_id, ads_endpoint, batch_size, batches_sent, rows_sent, rows_succeeded, rows_failed, retry_attemps, ads_load_status, load_started_at, load_completed_at, error_details)
            VALUES
            ('{period}', '{run_id}', 'orchestrator', '{ads_endpoint}', {batch_size}, 0, 0, 0, 0, 0, 'SKIPPED', current_timestamp(), current_timestamp(), 'NO_DATA')
        """)
        mssparkutils.notebook.exit("NO_DATA")
    
    # WARNING: Collect can consume significant memory with large datasets
    print(f"[nb_ads_load] Collecting tpir_load_equivalent data (WARNING: memory intensive for large datasets)")
    tpir_list = tpir.collect()
    total_rows = len(tpir_list)
    print(f"[nb_ads_load] Collected {total_rows} rows; planning {(total_rows + batch_size - 1) // batch_size} batches")
    
except Exception as e:
    print(f"[ERROR] Failed to load tpir_load_equivalent: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# ---------- Helper functions ----------
def send_batch(batch_data, attempt=1):
    """Send batch to ADS with retries."""
    try:
        response = requests.post(
            f"{ads_endpoint}/api/{api_version}/holdings/batch",
            json=batch_data,
            timeout=timeout_sec,
            headers={"Content-Type": "application/json"}
        )
        response.raise_for_status()
        result = response.json()
        return True, result
    except requests.exceptions.Timeout:
        if attempt < retry_max:
            print(f"[nb_ads_load] Timeout (attempt {attempt}/{retry_max}); retrying in {retry_backoff}s...")
            time.sleep(retry_backoff)
            return send_batch(batch_data, attempt + 1)
        else:
            return False, "Timeout after max retries"
    except requests.exceptions.HTTPError as e:
        status_code = e.response.status_code if e.response else "unknown"
        if attempt < retry_max and status_code in [503, 429, 502, 504]:  # Retryable HTTP errors
            print(f"[nb_ads_load] HTTP {status_code} (attempt {attempt}/{retry_max}); retrying in {retry_backoff}s...")
            time.sleep(retry_backoff)
            return send_batch(batch_data, attempt + 1)
        else:
            return False, f"HTTP Error {status_code}: {str(e)}"
    except Exception as e:
        if attempt < retry_max:
            print(f"[nb_ads_load] Batch failed (attempt {attempt}/{retry_max}): {type(e).__name__}: {e}. Retrying...")
            time.sleep(retry_backoff)
            return send_batch(batch_data, attempt + 1)
        else:
            return False, f"{type(e).__name__}: {str(e)}"

# ---------- Send batches ----------
try:
    batches_sent = 0
    rows_succeeded = 0
    rows_failed = 0
    errors = []
    
    for i in range(0, total_rows, batch_size):
        batch = tpir_list[i:i + batch_size]
        batch_index = (i // batch_size) + 1
        
        try:
            batch_data = []
            for row in batch:
                try:
                    batch_data.append({
                        "policy_id": str(row["policy_id"]) if row["policy_id"] else "",
                        "isin": str(row["isin"]) if row["isin"] else "",
                        "security_name": str(row["security_name"]) if row["security_name"] else "",
                        "quantity": float(row["quantity"] or 0),
                        "market_value_gbp": float(row["market_value_gbp"] or 0),
                        "currency": str(row["currency"]) if row["currency"] else "GBP",
                        "valuation_date": str(row["valuation_date"]) if row["valuation_date"] else None
                    })
                except (ValueError, TypeError) as e:
                    rows_failed += 1
                    errors.append(f"Row {i} parse error: {e}")
                    continue
            
            if not batch_data:
                continue
            
            success, result = send_batch(batch_data)
            batches_sent += 1
            
            if success:
                rows_succeeded += len(batch_data)
                print(f"[nb_ads_load] Batch {batch_index}: {len(batch_data)} rows sent successfully")
            else:
                rows_failed += len(batch_data)
                errors.append(f"Batch {batch_index}: {result}")
                print(f"[nb_ads_load] Batch {batch_index}: FAILED - {result}")
        
        except Exception as e:
            rows_failed += len(batch)
            errors.append(f"Batch {batch_index} processing error: {type(e).__name__}: {e}")
            print(f"[ERROR] Batch {batch_index} processing failed: {e}")
            continue
    
    # Determine status
    if rows_failed == 0 and rows_succeeded > 0:
        ads_status = "committed"
    elif rows_succeeded > 0:
        ads_status = "partial"
    else:
        ads_status = "failed"
    
    error_json = json.dumps(errors[-10:]) if errors else "[]"  # Keep last 10 errors
    
    # Write audit with error handling
    spark.sql(f"""
        INSERT INTO ads_load_audit
        (period, run_id, dfm_id, ads_endpoint, batch_size, batches_sent, rows_sent, rows_succeeded, rows_failed, retry_attemps, ads_load_status, load_started_at, load_completed_at, error_details)
        VALUES
        ('{period}', '{run_id}', 'orchestrator', '{ads_endpoint}', {batch_size}, {batches_sent}, {total_rows}, {rows_succeeded}, {rows_failed}, {retry_max}, '{ads_status}', current_timestamp(), current_timestamp(), '{error_json.replace("'", "''")}')
    """)
    
    print(f"[nb_ads_load] COMPLETE status={ads_status}, sent={rows_succeeded}/{total_rows}, failed={rows_failed}")
    if errors:
        print(f"[nb_ads_load] Errors recorded: {len(errors)} total")
    
    mssparkutils.notebook.exit(ads_status.upper())

except Exception as e:
    print(f"[ERROR] ADS load failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)